In [2]:
import os
import time
import re
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

# 1. Load Excel Data
input_file = "facebook pages.xlsx"
df = pd.read_excel(input_file)
df.columns = df.columns.str.strip()

processed_data = []

# 2. Setup Selenium Chrome Browser (Headless - background me chalega)
chrome_options = Options()
chrome_options.add_argument("--headless") # Browser window chupi rahegi
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--window-size=1920,1080")
chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

# Browser initiate karein
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

print("Starting Advanced Selenium Scraper...")

# 3. Process Each Page
for index, row in df.iterrows():
    url = row['page_link'].strip()
    page_name = row['Page_name'].strip()
    page_id = row['PAGE_ID']
    
    # URL Format settings
    url_with_videos = url
    if "profile.php" in url:
        if 'sk=videos' not in url:
            url_with_videos = url + '&sk=videos' if '?' in url else url + '?sk=videos'
    else:
        if not url.endswith('/videos'):
            url_with_videos = url.rstrip('/') + '/videos'
            
    print(f"\n--- Processing [{index+1}/{len(df)}]: {page_name} ---")
    
    try:
        driver.get(url_with_videos)
        time.sleep(5) # Page ko load hone ka time dein
        
        # Facebook page ko thoda niche scroll karein taaki videos load ho jayein
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight/2);")
        time.sleep(3)
        
        # Page ka HTML source nikalna
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        html_text = soup.get_text()
        
        # --- VIEWS & VIDEO COUNT EXTRACTION LOGIC ---
        # Facebook par views hamesha "K views", "M views" ya "views" text se pehle hote hain
        view_matches = re.findall(r'([\d\.]+ ?[KMB]?)\s*views', html_text, re.IGNORECASE)
        
        video_count = len(view_matches)
        total_views = 0
        
        for view_str in view_matches:
            view_str = view_str.upper().replace(' ', '').strip()
            try:
                if 'K' in view_str:
                    total_views += int(float(view_str.replace('K', '')) * 1000)
                elif 'M' in view_str:
                    total_views += int(float(view_str.replace('M', '')) * 1000000)
                else:
                    total_views += int(view_str)
            except:
                continue
                
        # Followers Count extract karne ki koshish (Page layout ke hisab se)
        followers = "N/A"
        follower_match = re.search(r'([\d\.]+ ?[KMB]?)\s*followers', html_text, re.IGNORECASE)
        if follower_match:
            followers = follower_match.group(1)

        print(f"✅ Success -> Videos Found: {video_count} | Total Views: {total_views} | Followers: {followers}")

    except Exception as e:
        print(f"❌ Error processing {page_name}: {e}")
        video_count, total_views, followers = 0, 0, "N/A"
        
    processed_data.append({
        'PAGE_ID': page_id,
        'Page_name': page_name,
        'page_link': url,
        'page followers': followers,
        'video posts_count': video_count,
        'page_views': total_views
    })

# 4. Save to Excel
driver.quit() # Browser close karein
output_df = pd.DataFrame(processed_data)
output_df.to_excel("facebook_pages_performance_report.xlsx", index=False)
print("\n All Pages Processed! Report saved as 'facebook_pages_performance_report.xlsx'")

Starting Advanced Selenium Scraper...

--- Processing [1/44]: Fruity AI ---
✅ Success -> Videos Found: 0 | Total Views: 0 | Followers: N/A

--- Processing [2/44]: Story Zone ---
✅ Success -> Videos Found: 4 | Total Views: 5326 | Followers: 10 

--- Processing [3/44]: Stories TIME ---
✅ Success -> Videos Found: 1 | Total Views: 1 | Followers: 4 

--- Processing [4/44]: Night Whispers ---
✅ Success -> Videos Found: 5 | Total Views: 94 | Followers: 0 

--- Processing [5/44]: Kevin Edits ---
✅ Success -> Videos Found: 7 | Total Views: 76 | Followers: 18 

--- Processing [6/44]: ibministories ---
✅ Success -> Videos Found: 6 | Total Views: 15 | Followers: 28 

--- Processing [7/44]: katmeowvibes ---
✅ Success -> Videos Found: 8 | Total Views: 25 | Followers: 18 

--- Processing [8/44]: Ibminis ---
✅ Success -> Videos Found: 0 | Total Views: 0 | Followers: N/A

--- Processing [9/44]: Story.here ---
✅ Success -> Videos Found: 3 | Total Views: 636 | Followers: 13 

--- Processing [10/44]: IB S